# Conversion Modeling

> **📦 Optional dependencies**
>
> The plots in this article use `plotly`, which ships with the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.
>
> The code is collapsed by default. Click **Show code** (or expand the cell in Jupyter) to see how the synthetic data is generated and analyzed.

Adaptive models in Pega Customer Decision Hub (CDH) usually learn from **engagement**: a customer clicks on a web banner, opens an e-mail, or accepts an offer in the contact center. Engagement is fast and plentiful, but it is a proxy. What the business really cares about is often the **conversion**: the customer actually buys the handset, opens the savings account, or takes out the mortgage.

Conversion modeling lets ADM learn from these downstream outcomes. This article takes a data science view of the approach: what a conversion is, how conversions are attributed to actions, when conversion modeling is (and is not) worth the effort, how far back to look, and how conversion and engagement propensities can be compared in one arbitration.

Conversion modeling is the natural choice where engagement feedback is missing or unreliable: e-mails and SMS messages without links, direct mail, or agent-assisted channels such as the call center. For products with a long customer journey, such as a mortgage, it is a conscious choice between driving towards the sale and nurturing longer-term engagement. In CDH, conversion modeling is currently available as a Preview Release. This article complements the product documentation, [Determining Next Best Actions by using conversions](https://docs.pega.com/bundle/customer-decision-hub/page/customer-decision-hub/cdh-portal/conversions-next-best-actions.html), which describes how to set it up.

- [What is a conversion?](#What-is-a-conversion?)
- [Attribution](#Attribution)
- [Do you need conversion modeling?](#Do-you-need-conversion-modeling?)
- [The lookback period](#The-lookback-period)
- [Mixing conversion and engagement](#Mixing-conversion-and-engagement)
- [Summary](#Summary)

In [1]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook separately

import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [2]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl

rng = np.random.default_rng(42)

## What is a conversion?

Conversion modeling is mostly a **business problem**, not a modeling problem. Technically, a conversion is just another outcome that ADM learns from. The hard part is deciding *what* that outcome is and what to attribute it to. That choice varies a lot by industry, product, and business goal:

| Domain | Possible conversion events |
|---|---|
| Retail banking | Current or savings account opened, credit card activated (not just approved), first deposit or first card transaction, salary deposit switched |
| Lending | Mortgage or loan application submitted, application approved, loan funded or disbursed |
| Insurance | Quote requested, policy bound, first premium paid, additional coverage added |
| Telecommunications | Handset purchased, contract upgraded or renewed, add-on (roaming, data bundle) activated, customer not churning after a retention offer |
| Retail and e-commerce | Order placed, order not returned within the return window, loyalty program sign-up |
| Digital self-service | App installed and logged in, paperless statements enabled, card added to a digital wallet |

Marketers often separate **macro conversions** (the actual sale, such as a funded mortgage) from **micro conversions**: smaller steps that signal intent, such as using the mortgage calculator on the website, starting an application, booking an appointment with an advisor, or downloading a brochure. Micro conversions occur more often and sooner, so models learn faster. But they are further removed from business value. Macro conversions are closer to value but rare and slow, so models learn slowly.

A few questions the business needs to answer before turning on conversion modeling:

- **Which event counts?** "Application started", "application approved" and "loan disbursed" are three different conversions, with different volumes and delays.
- **Which actions promote it?** Conversions are reported against a **conversion event ID**, usually the ID of the purchased product, not against an action. (It can also identify a journey that is not tied to a product, such as downloading an app.) Several actions, treatments, and channels can promote the same product. CDH uses a **product catalog** integration and a **product-to-Action mapping** to link a conversion to the actions that promote it.
- **Where does the signal come from?** Conversions usually happen outside CDH (in a store, a fulfillment system, a policy administration system). CDH supports two ways to get them in. The recommended one is a **conversion signal** sent by an external conversion tracking system through the Capture Conversion API, with the subject ID and the conversion event ID. The alternative is to **detect conversions from updates to the customer's product holdings** after a data import, for example a new product holding, a changed contract end date, or a changed price, as defined in the *Detect Conversion* decision data rule (DDR).
- **How long can it take?** This sets how long decision data must be kept so that a late conversion can still be matched to the decision that caused it. See [The lookback period](#The-lookback-period).
- **Would it have happened anyway?** Many conversions are organic. Product purchases happen whether or not an action was shown, so not every conversion is evidence that an action worked.

Pega's documentation describes the configuration steps in detail: [Determining Next Best Actions by using conversions](https://docs.pega.com/bundle/customer-decision-hub/page/customer-decision-hub/cdh-portal/conversions-next-best-actions.html).

## Attribution

Once a conversion is reported, it has to be linked to the actions that might have caused it. This is the **attribution problem**: given a conversion event, which of the earlier touchpoints (impressions of actions that promote the converted product) get credit, and how much?

Attribution is an old and largely unsolved problem in marketing. Approaches range from simple rules to data-driven methods such as Shapley values, Markov chain removal effects, and controlled uplift experiments. Pega does **not** try to solve attribution in a data-driven way. It offers a set of configurable, rule-based schemes. The decisioning architect picks the one that best matches the business view in the *ConversionAttribution* DDR. Every touchpoint that receives credit is then used as a positive example for the conversion model of that action, weighted by its credit. The weight is stored in the `pyOutcomeWeight` property of the interaction history.

The available schemes are:

- **First**: all credit goes to the first related action the customer saw.
- **Last**: all credit goes to the most recent related action before the conversion.
- **Linear**: credit is shared equally across all related actions.
- **Decay**: credit is shared across all related actions, with more weight for more recent ones. With a linear decay over $n$ touchpoints, touchpoint $i$ (oldest is 1) gets weight $\frac{i}{\sum_{j=1}^{n} j} = \frac{2i}{n(n+1)}$. For three touchpoints this is $\frac{1}{6}$, $\frac{2}{6}$, and $\frac{3}{6}$.
- **Custom**: you can plug in your own scheme.

Only touchpoints inside the **lookback period** (*Period in seconds* in the same DDR) are considered. The example below shows a customer who saw four actions for the same product over two weeks and then bought it.

In [3]:
touchpoints = ["E-mail<br>day 0", "Web<br>day 4", "App<br>day 8", "E-mail<br>day 11"]
n = len(touchpoints)
schemes = {
    "First": [1.0] + [0.0] * (n - 1),
    "Last": [0.0] * (n - 1) + [1.0],
    "Linear": [1 / n] * n,
    "Decay": [2 * (i + 1) / (n * (n + 1)) for i in range(n)],
}
attribution = pl.DataFrame(
    {
        "Scheme": [s for s in schemes for _ in touchpoints],
        "Touchpoint": touchpoints * len(schemes),
        "Credit": [c for credits in schemes.values() for c in credits],
    }
)

fig = px.bar(
    attribution,
    x="Touchpoint",
    y="Credit",
    facet_col="Scheme",
    color="Scheme",
    text_auto=".0%",
    title="Credit given to each touchpoint before a conversion on day 14",
    template="plotly_white",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(tickformat=".0%", range=[0, 1.15])
fig.update_traces(textangle=0, textposition="outside")
fig.update_xaxes(title_text="", tickangle=0, tickfont_size=10)
fig.update_layout(showlegend=False, height=420, margin=dict(t=90))
fig.show()

The choice of scheme matters less than you might think when customers mostly see a single action for a product. It matters more when products are promoted through many actions and channels. In that case First and Last reward very different parts of the journey (awareness vs. closing), while Linear and Decay spread the credit.

## Do you need conversion modeling?

Conversion modeling comes with real cost: an extra integration to report conversions, decision data that has to be kept for much longer, and models that learn from far fewer and much later signals. So before investing, check whether it would actually change any decisions.

The key point is that **CDH arbitration only cares about the ranking of actions**, not the absolute propensity values. If an action's conversion rate is roughly a fixed fraction of its engagement rate, then ranking actions by engagement already gives (almost) the same order as ranking them by conversion. In that case the engagement models already do the job, and the cheaper and faster engagement signal is the better choice.

You can check this with data you probably already have: for every action, compute the engagement rate (clicks or accepts per impression) and the conversion rate (conversions per impression) over the same period, and plot one against the other. The example below uses synthetic data for 200 actions.

In [4]:
n_actions = 200
impressions = rng.integers(20_000, 250_000, n_actions)
true_engagement = rng.lognormal(mean=np.log(0.02), sigma=0.55, size=n_actions)
true_conversion = true_engagement * 0.10 * rng.lognormal(mean=0.0, sigma=0.22, size=n_actions)
groups = rng.choice(["Cards", "Loans", "Savings", "Insurance"], n_actions)

actions = pl.DataFrame(
    {
        "Action": [f"Action {i + 1:03d}" for i in range(n_actions)],
        "Group": groups,
        "Impressions": impressions,
        "Engagements": rng.binomial(impressions, true_engagement),
        "Conversions": rng.binomial(impressions, true_conversion),
    }
).with_columns(
    EngagementRate=pl.col("Engagements") / pl.col("Impressions"),
    ConversionRate=pl.col("Conversions") / pl.col("Impressions"),
)

# Least squares fit through the origin: conversion rate = slope * engagement rate
slope = (actions["EngagementRate"] * actions["ConversionRate"]).sum() / (actions["EngagementRate"] ** 2).sum()
stats = actions.select(
    Pearson=pl.corr("EngagementRate", "ConversionRate"),
    Spearman=pl.corr("EngagementRate", "ConversionRate", method="spearman"),
)
pearson, spearman = stats.row(0)

fig = px.scatter(
    actions,
    x="EngagementRate",
    y="ConversionRate",
    size="Impressions",
    color="Group",
    hover_name="Action",
    size_max=14,
    opacity=0.7,
    hover_data={"Impressions": ":,", "EngagementRate": ":.2%", "ConversionRate": ":.3%"},
    template="plotly_white",
    title=(
        "Engagement vs. conversion rate per action<br>"
        f"<sup>Pearson r = {pearson:.2f}, Spearman rank correlation = {spearman:.2f}</sup>"
    ),
)
x_max = actions["EngagementRate"].max() * 1.05
fig.add_trace(
    go.Scatter(
        x=[0, x_max],
        y=[0, slope * x_max],
        mode="lines",
        line=dict(dash="dash", color="grey"),
        name=f"Fit: {slope:.1%}",
    )
)
fig.update_xaxes(tickformat=".1%", title_text="Engagement rate (engagements / impressions)", rangemode="tozero")
fig.update_yaxes(tickformat=".2%", title_text="Conversion rate (conversions / impressions)", rangemode="tozero")
fig.update_layout(height=580, margin=dict(l=90), legend=dict(title_text="", orientation="h", y=-0.18))
fig.show()

Each dot is an action, sized by its number of impressions. Here the dots lie close to a straight line through the origin: conversions are about one tenth of engagements, for every action. The rank correlation is high. Ranking actions by engagement propensity therefore gives almost the same order as ranking by conversion propensity, and conversion modeling would add little.

How to read such a plot for your own data:

- **Points close to a line through the origin**: engagement is a good proxy for conversion. Stick with engagement models. They learn faster and need no extra integration.
- **A wide spread, or clear clusters with different slopes**: some actions get a lot of clicks that do not turn into sales (think clickbait, or curiosity about a new product), while others convert well from few clicks. This is where conversion modeling can change the ranking and add business value.
- **Actions with engagement but no conversions** (on the x-axis): check first whether the conversions are reported and mapped to these actions before drawing conclusions.

This is an action-level view. A strong correlation across actions does not guarantee that the same holds within an action across customers. A natural next step is to bin the engagement propensities that ADM produced and check that the conversion rate also increases across the bins.

## The lookback period

Engagement usually arrives within seconds (a click) or days (an e-mail open). Conversions take longer: customers think about it, compare offers, visit a store, or wait for the next paycheck. To attribute a conversion to an action, CDH must keep the decision data (the customer's predictor values at decision time) until the conversion arrives. The **lookback period**, also called the attribution window, is how long CDH waits. It is a trade-off:

- **Too short**: genuine conversions arrive after the decision data is gone. The model never learns about them and the conversion rate is underestimated, especially for slower products.
- **Too long**: storage grows, models learn later, and more *organic* conversions (purchases that would have happened anyway, unrelated to the action) are credited to whatever action happened to be shown.

Choose the period from data: for every conversion, look up the most recent impression of an action that promotes the same product, and compute the delay between the two. All you need is the interaction history, with the impressions and the conversions. The example below simulates such a history for eight products over four months. Most conversions are driven by an impression and follow it within days. On top of that there is a small background of organic conversions.

In [5]:
sim_rng = np.random.default_rng(7)
start = np.datetime64("2026-01-01", "ms")
horizon_days = 120
n_customers = 50_000
products = [
    "Credit Card",
    "Personal Loan",
    "Savings Account",
    "Mortgage",
    "Car Insurance",
    "Home Insurance",
    "Travel Insurance",
    "Pension Plan",
]
channels = ["Web", "E-mail", "Mobile"]


def to_timestamps(days):
    return start + (days * 86_400_000).astype("timedelta64[ms]")


frames = []
for product in products:
    actions = [f"{product.replace(' ', '')}_{variant}" for variant in ("Promo", "Reminder", "CrossSell")]

    # Impressions: a subset of the customers sees one or more actions for this product
    reached = np.flatnonzero(sim_rng.random(n_customers) < sim_rng.uniform(0.3, 0.5))
    impression_customers = np.repeat(reached, 1 + sim_rng.poisson(1.0, reached.size))
    impression_days = sim_rng.uniform(0, horizon_days, impression_customers.size)
    frames.append(
        pl.DataFrame(
            {
                "CustomerID": impression_customers,
                "Action": sim_rng.choice(actions, impression_customers.size),
                "Channel": sim_rng.choice(channels, impression_customers.size),
                "OutcomeTime": to_timestamps(impression_days),
            }
        ).with_columns(Product=pl.lit(product), Outcome=pl.lit("Impression"))
    )

    # Some impressions lead to a conversion, after a gamma distributed delay with a mean of about 5 days
    driven = sim_rng.random(impression_customers.size) < 0.03
    driven_days = impression_days[driven] + sim_rng.gamma(2.0, sim_rng.uniform(2.0, 3.0), driven.sum())
    # Organic conversions happen anyway, at random moments, regardless of any impression
    organic_customers = sim_rng.integers(0, n_customers, n_customers // 100)
    organic_days = sim_rng.uniform(0, horizon_days, organic_customers.size)
    frames.append(
        pl.DataFrame(
            {
                "CustomerID": np.concatenate([impression_customers[driven], organic_customers]),
                "OutcomeTime": to_timestamps(np.concatenate([driven_days, organic_days])),
            }
        ).with_columns(Product=pl.lit(product), Outcome=pl.lit("Conversion"))
    )

# Conversions have no action or channel: they are reported against the product
ih = (
    pl.concat(frames, how="diagonal")
    .filter(pl.col("OutcomeTime") < to_timestamps(np.array([horizon_days]))[0])
    .with_columns(
        CustomerID=pl.col("CustomerID") + 1_000_000,
        OutcomeTime=pl.col("OutcomeTime").dt.truncate("1s"),
    )
    .select("CustomerID", "Action", "Product", "Channel", "OutcomeTime", "Outcome")
    .sort("OutcomeTime")
)

# Show the journey of one customer who converted after seeing several actions for the same product
example = (
    ih.group_by("CustomerID", "Product")
    .agg(
        Impressions=(pl.col("Outcome") == "Impression").sum(),
        Conversions=(pl.col("Outcome") == "Conversion").sum(),
        LastImpression=pl.col("OutcomeTime").filter(pl.col("Outcome") == "Impression").max(),
        ConversionTime=pl.col("OutcomeTime").filter(pl.col("Outcome") == "Conversion").min(),
    )
    .filter(
        pl.col("Impressions") == 3,
        pl.col("Conversions") == 1,
        pl.col("LastImpression") < pl.col("ConversionTime"),
    )
    .sort("CustomerID")
    .row(0, named=True)
)
ih.filter(pl.col("CustomerID") == example["CustomerID"], pl.col("Product") == example["Product"])

CustomerID,Action,Product,Channel,OutcomeTime,Outcome
i64,str,str,str,datetime[ms],str
1000064,"""HomeInsurance_Reminder""","""Home Insurance""","""Mobile""",2026-01-17 19:56:14,"""Impression"""
1000064,"""HomeInsurance_Promo""","""Home Insurance""","""Mobile""",2026-02-17 19:27:50,"""Impression"""
1000064,"""HomeInsurance_Reminder""","""Home Insurance""","""Web""",2026-03-20 09:31:17,"""Impression"""
1000064,null,"""Home Insurance""",null,2026-03-22 17:36:17,"""Conversion"""


Above is a small piece of the simulated interaction history: one customer's impressions of the actions that promote a product, followed by a conversion of that product. Conversions are reported against the product (the conversion event ID), not against an action. That's why the `Action` and `Channel` columns are empty for them, just like the raw conversion records that CDH writes to the interaction history.

To measure the delays, take the first conversion per customer and product, and look up the most recent impression of the same product before it. An as-of join does exactly that. Conversions without any earlier impression are organic by definition and are left out.

In [6]:
max_days = 45

impressions = (
    ih.filter(pl.col("Outcome") == "Impression")
    .select("CustomerID", "Product", ImpressionTime="OutcomeTime")
    .sort("ImpressionTime")
)
conversions = (
    ih.filter(pl.col("Outcome") == "Conversion")
    .group_by("CustomerID", "Product")
    .agg(ConversionTime=pl.col("OutcomeTime").min())
    .sort("ConversionTime")
)
delays = conversions.join_asof(
    impressions,
    left_on="ConversionTime",
    right_on="ImpressionTime",
    by=["CustomerID", "Product"],
    strategy="backward",
    check_sortedness=False,  # sorted on time; the per-customer groups are therefore sorted too
).with_columns(Days=(pl.col("ConversionTime") - pl.col("ImpressionTime")).dt.total_days())
without_impression = delays["ImpressionTime"].null_count() / delays.height

conversion_delays = (
    delays.drop_nulls("Days")
    .filter(pl.col("Days") < max_days)
    .group_by("Product", "Days")
    .agg(Conversions=pl.len())
    .sort("Product", "Days")
    .with_columns(Share=pl.col("Conversions") / pl.col("Conversions").sum().over("Product"))
    .with_columns(Cumulative=pl.col("Share").cum_sum().over("Product"))
)
summary = (
    conversion_delays.group_by("Days")
    .agg(
        pl.col("Share").mean().alias("Share"),
        pl.col("Share").std().alias("ShareStd"),
        pl.col("Cumulative").mean().alias("Cumulative"),
        pl.col("Cumulative").std().alias("CumulativeStd"),
    )
    .sort("Days")
)
within_10 = summary.filter(pl.col("Days") == 9)["Cumulative"].item()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=summary["Days"],
        y=summary["Share"],
        error_y=dict(type="data", array=summary["ShareStd"], thickness=1, width=3),
        mode="lines+markers",
        name="Conversions on that day",
        marker_size=5,
    )
)
fig.add_trace(
    go.Scatter(
        x=summary["Days"],
        y=summary["Cumulative"],
        error_y=dict(type="data", array=summary["CumulativeStd"], thickness=1, width=3),
        mode="lines+markers",
        name="Cumulative",
        yaxis="y2",
        marker_size=5,
    )
)
fig.add_vline(
    x=9.5,
    line_dash="dash",
    line_color="grey",
    annotation_text=f"10 days: {within_10:.0%} of conversions",
    annotation_position="top right",
)
fig.update_layout(
    template="plotly_white",
    title=(
        "Days between last impression and conversion<br>"
        f"<sup>Mean and standard deviation across {len(products)} products. "
        f"<br>{without_impression:.0%} of the conversions had no earlier impression and are left out.</sup>"
    ),
    xaxis=dict(title="Days since last impression of a related action"),
    yaxis=dict(title="Share of conversions per day", tickformat=".0%", rangemode="tozero"),
    yaxis2=dict(title="Cumulative share", tickformat=".0%", overlaying="y", side="right", range=[0, 1.05]),
    legend=dict(orientation="h", y=-0.2),
    height=570,
    margin=dict(l=90, r=120, t=120),
)
fig.show()

In this example, most conversions happen within about 10 days of the last impression. After that, the daily share drops to a low, flat level. That flat tail is what you would expect from organic conversions that have nothing to do with the action. Extending the lookback period well beyond the point where the curve flattens mostly adds noise (and storage cost), not signal. In this example a lookback period of about 10 to 14 days is a sensible choice.

Some practical notes for running this analysis on your own data:

- Compare the delay curve against a **baseline**: conversions of customers who did *not* see a related action. The flat tail should match that baseline. Delays where the curve is clearly above the baseline are where the actions have an effect.
- Look at the curves **per product or product group**. A credit card and a mortgage can have very different delays, and the lookback period has to cover the slowest product you want to model.
- Dates without a time component make same-day ordering ambiguous. If the impression and the conversion are on the same day, you may not know which came first.

## Mixing conversion and engagement

In practice, not every action has a conversion model that can be used. Some actions have no conversion at all (a "happy birthday" message, a service notification). Others are new, and their conversion model has not seen enough positives yet. These actions keep using their engagement model. The pre-arbitration set of actions therefore contains a mix: some actions have a conversion propensity, others an engagement propensity. Since conversion rates are typically an order of magnitude lower than engagement rates, these numbers cannot be compared directly.

### Two propensities per action

For every action $a$ and customer $x$, CDH runs two adaptive models side by side:

- the **engagement model**, which estimates $p^{E}_{a}(x) = P(\text{engage} \mid \text{impression}, x)$ and is trained on clicks or accepts, as usual;
- the **conversion model**, which estimates $p^{C}_{a}(x) = P(\text{convert} \mid \text{impression}, x)$ and is trained on the attributed conversions, where each impression gets a weight between 0 and 1 from the attribution scheme.

The conversion model is a copy of the engagement model with the outcome labels *Conversion* and *No conversion*. Both are combined in an engagement and conversion strategy (based on the *EngagementAndConversionTemplate* strategy). For customers in the conversion group (see below), it uses the conversion propensity when the conversion model is mature enough. Otherwise it uses the engagement propensity.

### Scaling

To make a conversion propensity comparable to engagement propensities, CDH multiplies it by a per-action scaling factor: the ratio between the action's overall engagement rate and its overall conversion rate:

$$
s_a = \frac{\bar{p}^{E}_{a}}{\bar{p}^{C}_{a}} = \frac{\text{engagements}_a / \text{impressions}_a}{\text{conversions}_a / \text{impressions}_a} = \frac{\text{engagements}_a}{\text{conversions}_a}
\qquad\qquad
\tilde{p}^{C}_{a}(x) = s_a \cdot p^{C}_{a}(x)
$$

Example: an action for a handset has an overall engagement rate of 1.6% and an overall conversion rate of 0.2%, so $s_a = 8$. For a particular customer the conversion model gives 0.15%, which scales to $8 \times 0.15\% = 1.2\%$. That can now be compared with the 1.1% engagement propensity of another action, for which no conversion model is available yet.

The scaling factor is the same for all customers. So it does not change the ranking of customers *within* an action. It only puts the action on the same scale as the others. If engagement and conversion are as closely related as in the scatter plot above, $s_a$ is about 10 for every action, and the scaled conversion propensity is close to the engagement propensity. That is another way of seeing that conversion modeling then changes little.

### It is really the value that is scaled

The CDH priority multiplies the propensity with the action's value (along with context weights and business levers, which we leave out here):

$$
\text{Priority}_a(x) = p_a(x) \cdot V_a
$$

Arbitration compares actions on the expected value **per impression**. For an engagement-driven action, $V_a$ is effectively the value of an engagement. For a conversion-driven action, the natural value is the value of a conversion, $V^{C}_{a}$, for example the margin on the product, and the expected value of an impression is $p^{C}_{a}(x) \cdot V^{C}_{a}$.

Reorder the scaled priority and the scaling factor ends up on the value instead of on the propensity:

$$
\underbrace{\left(s_a \cdot p^{C}_{a}(x)\right)}_{\text{scaled propensity}} \cdot V_a
\;=\;
p^{C}_{a}(x) \cdot \underbrace{\left(s_a \cdot V_a\right)}_{V^{C}_{a}}
\qquad\Longleftrightarrow\qquad
V_a = V^{C}_{a} \cdot \frac{\text{conversions}_a}{\text{engagements}_a}
$$

So the scaling does not really "inflate" the conversion propensity. It translates the value of a conversion (large: a product sale) into the value of a single engagement (much smaller: only about one in $s_a$ engagements turns into a sale). Everything in the arbitration is then expressed in the same unit, the value of an engagement, and conversion-driven and engagement-driven actions compete fairly.

### Test and learn

Optimizing for conversion does not always produce a better business result than optimizing for engagement. The engagement models learn much faster, and the conversion signal can be sparse, delayed, or noisy. CDH therefore splits customers into a **conversion group** and an **engagement group**, based on a hash of the customer ID. By default 25% of the customers get the conversion models and 75% stay on engagement models; the split can be changed in the *SetSubjectHashExt* extension strategy. Comparing conversions and other KPIs in both groups shows whether conversion modeling actually pays off. The lookback period and attribution scheme can be tuned based on these results.

## Summary

- **What a conversion is depends on the business.** It can be a purchase, an activation, an application, or a micro conversion such as using a mortgage calculator. Decide this first, together with how conversions are reported and which actions promote which products.
- **Attribution** links a conversion to earlier impressions. CDH offers First, Last, Linear, Decay, and Custom schemes. It does not try to solve attribution in a data-driven way.
- **Check whether you need it.** If conversion rates are roughly proportional to engagement rates across actions, the ranking hardly changes and engagement models are the cheaper, faster choice.
- **Pick the lookback period from data**: long enough to capture most of the impression-driven conversions, but not so long that organic conversions and storage costs dominate.
- **Scaling** puts conversion and engagement propensities on the same scale. You can also read it as converting the value of a conversion into the value of an engagement, so all actions are compared in the same unit.

## Further reading

- [Determining Next Best Actions by using conversions](https://docs.pega.com/bundle/customer-decision-hub/page/customer-decision-hub/cdh-portal/conversions-next-best-actions.html): overview, use cases, and implementation approaches in the Pega documentation.
- [Configuring the Conversion Attribution Decision Data Rule](https://docs.pega.com/bundle/customer-decision-hub/page/customer-decision-hub/cdh-portal/configuring-conversion-attribution.html): lookback period and attribution schemes.
- [Enabling conversion modeling for Next Best Actions](https://docs.pega.com/bundle/customer-decision-hub/page/customer-decision-hub/cdh-portal/enabling-conversion-modeling.html): conversion models, the engagement and conversion strategy, and the conversion/engagement groups.